## Final Results — RITMO vs Todas las Técnicas

Todas las técnicas de tokenización con K óptimo HMM por dataset, para todos los horizontes (96, 192, 336, 720).

**Resumible**: salta experimentos ya completados (metrics.npy en disco).

In [1]:
import os, sys, time, subprocess
import numpy as np

while not os.path.exists("run.py"):
    os.chdir("..")
sys.path.insert(0, ".")

PRED_LENS = [96, 192, 336, 720]
NON_HMM_TECHNIQUES = ["discretization", "text_based", "patching", "decomposition", "foundation"]

# Optimal HMM config per dataset (from K sweep)
# hmm_configs: list of (technique, K) — supports ties
DATASETS = {
    "ETTh1": {
        "root_path": "./dataset/ETT-small/",
        "data_path": "ETTh1.csv",
        "data": "ETTh1",
        "target": "OT",
        "hmm_configs": [("hmm_soft_residual", 4)],
    },
    "ETTh2": {
        "root_path": "./dataset/ETT-small/",
        "data_path": "ETTh2.csv",
        "data": "ETTh2",
        "target": "OT",
        "hmm_configs": [("hmm_soft", 6), ("hmm_soft", 8)],
    },
    "Weather": {
        "root_path": "./dataset/weather/",
        "data_path": "weather.csv",
        "data": "Weather",
        "target": "OT",
        "hmm_configs": [("hmm_soft", 8)],
    },
    "Electricity": {
        "root_path": "./dataset/electricity/",
        "data_path": "electricity.csv",
        "data": "custom",
        "target": "OT",
        "hmm_configs": [("hmm_soft_residual", 3)],
    },
}

TRANSFORMER_CFG = {
    "seq_len": 96,
    "label_len": 48,
    "d_model": 64,
    "n_heads": 4,
    "e_layers": 2,
    "d_ff": 128,
    "dropout": 0.1,
    "batch_size": 32,
    "learning_rate": 0.001,
    "lradj": "type1",
    "train_epochs": 10,
    "patience": 3,
}

total = sum(
    (len(ds["hmm_configs"]) + len(NON_HMM_TECHNIQUES)) * len(PRED_LENS)
    for ds in DATASETS.values()
)
print(f"Total experimentos: {total}")
print(f"Datasets: {list(DATASETS.keys())}")
print(f"Horizontes: {PRED_LENS}")
print(f"Tecnicas no-HMM: {NON_HMM_TECHNIQUES}")
for ds_name, ds_cfg in DATASETS.items():
    cfgs = ds_cfg["hmm_configs"]
    print(f"  {ds_name} HMM configs: {cfgs}")

Total experimentos: 100
Datasets: ['ETTh1', 'ETTh2', 'Weather', 'Electricity']
Horizontes: [96, 192, 336, 720]
Tecnicas no-HMM: ['discretization', 'text_based', 'patching', 'decomposition', 'foundation']
  ETTh1 HMM configs: [('hmm_soft_residual', 4)]
  ETTh2 HMM configs: [('hmm_soft', 6), ('hmm_soft', 8)]
  Weather HMM configs: [('hmm_soft', 8)]
  Electricity HMM configs: [('hmm_soft_residual', 3)]


In [2]:
def make_des(technique, K=None):
    return f"final_{technique}_K{K}" if K is not None else f"final_{technique}"

def result_exists(ds_cfg, pred_len, technique, K=None):
    des = make_des(technique, K)
    data = ds_cfg["data"]
    seq = TRANSFORMER_CFG["seq_len"]
    ll = TRANSFORMER_CFG["label_len"]
    dm = TRANSFORMER_CFG["d_model"]
    nh = TRANSFORMER_CFG["n_heads"]
    el = TRANSFORMER_CFG["e_layers"]
    df = TRANSFORMER_CFG["d_ff"]
    setting = (
        f"plan_a_{data}_96_{pred_len}_TransformerCommon_{data}"
        f"_ftS_sl{seq}_ll{ll}_pl{pred_len}"
        f"_dm{dm}_nh{nh}_el{el}_dl1"
        f"_df{df}_expand2_dc4_fc1_ebtimeF_dtTrue_{des}_0"
    )
    path = f"./results/{setting}/metrics.npy"
    return os.path.exists(path), path

def run_experiment(ds_cfg, pred_len, technique, K=None):
    des = make_des(technique, K)
    data = ds_cfg["data"]
    seq = str(TRANSFORMER_CFG["seq_len"])
    ll = str(TRANSFORMER_CFG["label_len"])
    dm = str(TRANSFORMER_CFG["d_model"])
    nh = str(TRANSFORMER_CFG["n_heads"])
    el = str(TRANSFORMER_CFG["e_layers"])
    df = str(TRANSFORMER_CFG["d_ff"])
    dropout = str(TRANSFORMER_CFG["dropout"])
    bs = str(TRANSFORMER_CFG["batch_size"])
    lr = str(TRANSFORMER_CFG["learning_rate"])
    lradj = TRANSFORMER_CFG["lradj"]
    epochs = str(TRANSFORMER_CFG["train_epochs"])
    patience = str(TRANSFORMER_CFG["patience"])
    cmd = [
        "python", "-u", "run.py",
        "--task_name", "plan_a",
        "--is_training", "1",
        "--root_path", ds_cfg["root_path"],
        "--data_path", ds_cfg["data_path"],
        "--model_id", f"{data}_96_{pred_len}",
        "--model", "TransformerCommon",
        "--data", data,
        "--features", "S",
        "--target", ds_cfg["target"],
        "--seq_len", seq,
        "--label_len", ll,
        "--pred_len", str(pred_len),
        "--enc_in", "1",
        "--dec_in", "1",
        "--c_out", "1",
        "--d_model", dm,
        "--n_heads", nh,
        "--e_layers", el,
        "--d_ff", df,
        "--dropout", dropout,
        "--batch_size", bs,
        "--learning_rate", lr,
        "--lradj", lradj,
        "--train_epochs", epochs,
        "--patience", patience,
        "--use_gpu", "0",
        "--technique", technique,
        "--des", des,
        "--itr", "1",
    ]
    if K is not None:
        cmd += ["--hmm_k", str(K)]

    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=3600)
    elapsed = time.time() - t0

    mse_line = [l for l in proc.stdout.split("\n") if l.startswith("mse:")]
    if mse_line:
        return elapsed, mse_line[-1]
    err = proc.stderr[-500:] if proc.stderr else proc.stdout[-500:]
    return elapsed, f"ERROR: {err}"


# Build full experiment list
experiments = []
for ds_name, ds_cfg in DATASETS.items():
    for pred_len in PRED_LENS:
        for technique in NON_HMM_TECHNIQUES:
            experiments.append((ds_name, ds_cfg, pred_len, technique, None))
        for technique, K in ds_cfg["hmm_configs"]:
            experiments.append((ds_name, ds_cfg, pred_len, technique, K))

total = len(experiments)
done_count = sum(1 for (_, ds_cfg, pl, tec, K) in experiments if result_exists(ds_cfg, pl, tec, K)[0])
print(f"Completados: {done_count}/{total}")

t_start = time.time()
for i, (ds_name, ds_cfg, pred_len, technique, K) in enumerate(experiments, 1):
    exists, mpath = result_exists(ds_cfg, pred_len, technique, K)
    tag = make_des(technique, K)

    if exists:
        metrics = np.load(mpath)
        print(f"[{i:>3}/{total}] SKIP {ds_name} pl={pred_len} {tag} | MSE={metrics[0]:.6f}")
        continue

    print(f"[{i:>3}/{total}] RUN  {ds_name} pl={pred_len} {tag} ...", end=" ", flush=True)
    elapsed, result = run_experiment(ds_cfg, pred_len, technique, K)
    print(f"{elapsed:.0f}s | {result}")

elapsed_total = (time.time() - t_start) / 60
print(f"\nTotal elapsed: {elapsed_total:.1f}min")

Completados: 89/100
[  1/100] SKIP ETTh1 pl=96 final_discretization | MSE=0.187751
[  2/100] SKIP ETTh1 pl=96 final_text_based | MSE=0.184707
[  3/100] SKIP ETTh1 pl=96 final_patching | MSE=0.179315
[  4/100] SKIP ETTh1 pl=96 final_decomposition | MSE=0.182217
[  5/100] SKIP ETTh1 pl=96 final_foundation | MSE=0.179422
[  6/100] SKIP ETTh1 pl=96 final_hmm_soft_residual_K4 | MSE=0.181731
[  7/100] SKIP ETTh1 pl=192 final_discretization | MSE=0.215260
[  8/100] SKIP ETTh1 pl=192 final_text_based | MSE=0.214385
[  9/100] SKIP ETTh1 pl=192 final_patching | MSE=0.207290
[ 10/100] SKIP ETTh1 pl=192 final_decomposition | MSE=0.210995
[ 11/100] SKIP ETTh1 pl=192 final_foundation | MSE=0.207285
[ 12/100] SKIP ETTh1 pl=192 final_hmm_soft_residual_K4 | MSE=0.208215
[ 13/100] SKIP ETTh1 pl=336 final_discretization | MSE=0.235900
[ 14/100] SKIP ETTh1 pl=336 final_text_based | MSE=0.238006
[ 15/100] SKIP ETTh1 pl=336 final_patching | MSE=0.232773
[ 16/100] SKIP ETTh1 pl=336 final_decomposition | MSE=

In [3]:
import numpy as np, os, re
from collections import defaultdict

PRED_LENS = [96, 192, 336, 720]

rows = []
results_dir = "./results"
for d in os.listdir(results_dir):
    m = re.search(r"_final_([\w]+?)(?:_K(\d+))?_0$", d)
    ds_m = re.match(r"plan_a_(\w+?)_96_(\d+)_", d)
    if not m or not ds_m:
        continue
    mpath = os.path.join(results_dir, d, "metrics.npy")
    if not os.path.exists(mpath):
        continue
    k_part = f"_K{m.group(2)}" if m.group(2) else ""
    technique = m.group(1) + k_part
    dataset = ds_m.group(1)
    pred_len = int(ds_m.group(2))
    metrics = np.load(mpath)
    rows.append((dataset, technique, pred_len, float(metrics[0]), float(metrics[1])))

agg = defaultdict(lambda: defaultdict(list))
for dataset, technique, pred_len, mse, mae in rows:
    agg[dataset][technique].append((pred_len, mse, mae))

for ds in ["ETTh1", "ETTh2", "Weather", "custom"]:
    if ds not in agg:
        print(f"\n=== {ds} === (sin resultados)")
        continue
    print(f"\n=== {ds} ===")
    header = f"{'Technique':<32}"
    for pl in PRED_LENS:
        header += f" {'pl='+str(pl):>8}"
    header += f" {'AVG':>8}"
    print(header)
    techs = sorted(agg[ds].items(), key=lambda x: np.nanmean([r[1] for r in x[1]]))
    for technique, results in techs:
        by_pl = {pl: mse for pl, mse, mae in results}
        vals = [by_pl.get(pl, float("nan")) for pl in PRED_LENS]
        avg = np.nanmean(vals)
        row = f"{technique:<32}"
        for v in vals:
            row += f" {v:>8.4f}" if not np.isnan(v) else f" {'N/A':>8}"
        row += f" {avg:>8.4f}"
        print(row)


=== ETTh1 ===
Technique                           pl=96   pl=192   pl=336   pl=720      AVG
patching                           0.1793   0.2073   0.2328   0.2390   0.2146
foundation                         0.1794   0.2073   0.2327   0.2393   0.2147
hmm_soft_residual_K4               0.1817   0.2082   0.2328   0.2371   0.2149
decomposition                      0.1822   0.2110   0.2314   0.2382   0.2157
text_based                         0.1847   0.2144   0.2380   0.2389   0.2190
discretization                     0.1878   0.2153   0.2359   0.2484   0.2218

=== ETTh2 ===
Technique                           pl=96   pl=192   pl=336   pl=720      AVG
decomposition                      0.2922   0.3369   0.3711   0.3694   0.3424
text_based                         0.2851   0.3368   0.3830   0.3871   0.3480
foundation                         0.2877   0.3447   0.3837   0.3873   0.3509
patching                           0.2876   0.3450   0.3838   0.3872   0.3509
hmm_soft_K6                       